# The Mandelbrot set: an image from one line of arithmetic

The rule is $z \rightarrow z^2 + c$, started at $z = 0$. Pick a complex number `c`, apply
the rule forever, and ask a single question: does the sequence stay bounded? The set of
`c` for which it does is the Mandelbrot set.

That is the entire definition. Everything you are about to see — the spirals, the
seahorses, the endlessly repeating detail — follows from squaring and adding.

This project is the numerical-methods counterpart to the other two. Notebook 045 was about
data you did not control; notebook 046 was about a model whose output you could not
predict. This one is about the *arithmetic itself*: what it can represent, how fast it can
be made to run, and where it silently fails.

An interactive C++/SFML implementation of the same idea lives at
[tonigineer/mandelbrot-set](https://github.com/tonigineer/mandelbrot-set), which inspired
this notebook. None of its code is used here — it carries no licence, and this course needs
something that runs in a notebook without a compiler.

## Learning objectives

By the end, you can:

- turn a mathematical definition into a computable approximation and say what was lost;
- replace a loop over pixels with an array operation and measure the speed-up;
- explain what floating-point precision costs you at the limit; and
- justify a colour scale rather than picking the one that looked nice.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "README.md").exists(), "Open the course project folder first."

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from climate_course.mandelbrot import (
    DEEPEST_USEFUL_SPAN,
    MISIUREWICZ_POINT,
    REGIONS,
    in_set,
    iterations_for_span,
    render,
    smallest_reliable_span,
    to_rgb,
    zoom_spans,
)

for name, spec in REGIONS.items():
    print(f"{name:16s} span {spec['span']:<9g} {spec['description']}")

## 1. The definition is not computable

"Stays bounded forever" cannot be checked by a computer. Two substitutions make it
computable, and each one costs something:

1. **Iterate a finite number of times.** A point that has not escaped within the budget is
   *assumed* to be in the set. Raise the budget and some of those points turn out to escape
   after all, so the interior can only ever shrink.
2. **Declare escape at a finite radius.** Once $|z| > 2$ the sequence provably runs away,
   so any radius above 2 is safe.

Start with one point at a time, the way the definition reads.

In [ ]:
def escapes(c, max_iterations=200, escape_radius=2.0):
    """Return the iteration at which c escapes, or None if it never does."""
    z = 0
    for iteration in range(max_iterations):
        z = z * z + c
        if abs(z) > escape_radius:
            return iteration
    return None

for c in [0, -1, 0.25, -2, 1, 0.3 + 0.5j, -0.75 + 0.1j]:
    verdict = escapes(c)
    print(f"c = {str(c):>12}   " +
          ("stayed bounded for 200 iterations" if verdict is None
           else f"escaped at iteration {verdict}"))

- `c = 0.25` is on the boundary. Does 200 iterations settle the question? Try 20000. **TODO**
- `c = -2` is the leftmost point of the set. What does its orbit actually do? **TODO**
- Which of the two substitutions above could make a point *wrongly* appear to be in the
  set, and which could never? **TODO**

### A point that is exactly on the boundary

Most "boundary points" quoted in the literature are decimal approximations, and an
approximation is not on the boundary. `c = i` is different: it is exactly representable in
binary, and its orbit is *pre-periodic* — it reaches a cycle and stays there forever.

In [ ]:
z = 0j
orbit = [z]
for _ in range(8):
    z = z * z + MISIUREWICZ_POINT
    orbit.append(z)

print("orbit of c = i:")
print("  " + " -> ".join(f"{value:.0f}" for value in orbit))

z = 0j
for _ in range(20000):
    z = z * z + MISIUREWICZ_POINT
print(f"\nafter 20000 iterations |z| = {abs(z):.6f}  (exactly sqrt(2))")

# The eight-decimal value from the literature is close, but not on the boundary.
approximate = complex(-0.77568377, 0.13646737)
print(f"a literature point escapes at iteration {escapes(approximate, 2000)}")

The exact point orbits forever; the eight-decimal approximation escapes after a few hundred
iterations. Both are "on the boundary" to the precision anyone quotes them.

- Being *near* the boundary means escaping *slowly*. Why does that make the boundary the
  most expensive part of the picture to compute? **TODO**

## 2. From one point to an image, without a loop over pixels

A 600 by 450 image is 270,000 points. A Python loop over them is unbearably slow. The fix
is to advance every point at once, as arrays — the same move as Monday's NumPy session,
with something at stake.

Time both on a deliberately small image.

In [ ]:
import time

def escape_grid_loop(width, height, max_iterations=80):
    """The obvious implementation: one Python loop per pixel."""
    real = np.linspace(-2.1, 1.1, width)
    imaginary = np.linspace(-1.2, 1.2, height)
    out = np.zeros((height, width))
    for row in range(height):
        for column in range(width):
            c = real[column] + 1j * imaginary[row]
            z = 0j
            for iteration in range(max_iterations):
                z = z * z + c
                if abs(z) > 2:
                    out[row, column] = iteration
                    break
    return out

start = time.perf_counter()
escape_grid_loop(160, 120)
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
render(center=(-0.5, 0.0), span=3.2, width=160, height=120, max_iterations=80)
array_seconds = time.perf_counter() - start

print(f"pixel-by-pixel : {loop_seconds:7.3f} s")
print(f"whole-array    : {array_seconds:7.3f} s")
print(f"speed-up       : {loop_seconds / array_seconds:7.1f}x")
print(f"\nA 600x450 image the slow way would take about "
      f"{loop_seconds * (600 * 450) / (160 * 120):.0f} s.")

Open `render` in `src/climate_course/mandelbrot.py`. It does one thing beyond the obvious
vectorisation: it keeps a boolean `active` mask and only squares the points that have not
escaped yet.

- Most of the plane escapes within a handful of iterations. What does that do to the size
  of the arrays being squared after, say, twenty iterations? **TODO**
- The naive cost is `width * height * max_iterations`. Why is the real cost far lower? **TODO**

## 3. Colour is a decision, not a default

Escape counts near the boundary grow without limit, so mapping them linearly onto a colour
scale puts almost every pixel at one end. `to_rgb` does two things:

- takes a **square root**, compressing the range; and
- wraps the result through a **cyclic** colormap every few units, so the repeating bands
  are contour lines of escape time.

Interior points have no escape count at all and are painted separately.

In [ ]:
counts = render(center=(-0.5, 0.0), span=3.2, width=560, height=420)

finite = counts[np.isfinite(counts)]
print(f"escape counts: min {finite.min():.1f}, median {np.median(finite):.1f}, "
      f"max {finite.max():.1f}")
print(f"interior: {in_set(counts).mean():.1%} of the image")

overview = go.Figure(go.Image(z=to_rgb(counts)))
overview.update_layout(
    title="The Mandelbrot set",
    height=560, margin=dict(l=0, r=0, t=48, b=0),
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
)
overview

In [ ]:
# Try the alternatives before accepting the default.
# One full sweep of the colormap instead of repeating bands: set the cycle length to
# the largest value present, so nothing wraps.
one_sweep = float(np.sqrt(finite.max()))

comparison = {
    "cyclic bands (the default)": to_rgb(counts),
    "one sweep, cyclic map": to_rgb(counts, cycle=one_sweep),
    "one sweep, sequential map": to_rgb(counts, cycle=one_sweep, colormap="magma"),
}
for label, image in comparison.items():
    print(f"{label:28s} distinct colours: "
          f"{len(np.unique(image.reshape(-1, 3), axis=0)):,}")

figure = go.Figure(go.Image(z=comparison["one sweep, sequential map"]))
figure.update_layout(title="One sweep of a sequential colormap, no repeating bands",
                     height=520, margin=dict(l=0, r=0, t=48, b=0),
                     xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"))
figure

- Which choice makes the filaments legible, and which flattens them? **TODO**
- `conventions/figure-conventions.md` bans rainbow colormaps for data. Does that rule apply
  here? Say what the colour is encoding and whether a reader needs to decode it
  quantitatively. **TODO**

That last question is the real one. In notebook 045 the colour *was* the measurement and
had to be perceptually uniform. Here it is a contour scheme over a quantity nobody reads
off the image — a different job, and so a different rule.

## 4. Zooming

Two things must change together as you magnify:

- the **span** shrinks geometrically, so each frame magnifies by the same factor; and
- the **iteration budget** grows, because detail near the boundary takes longer to resolve.

Hold the budget fixed and a deep zoom fills with solid interior — the picture stops being
about the set and starts being about your budget.

In [ ]:
spans = zoom_spans(REGIONS["overview"]["span"], DEEPEST_USEFUL_SPAN, 8)
print(" span        magnification   iterations   interior")
for span in spans:
    frame = render(center=REGIONS["deep_spiral"]["center"], span=float(span),
                   width=200, height=150, max_iterations=iterations_for_span(float(span)))
    print(f"{span:9.2e}   {REGIONS['overview']['span'] / span:12,.0f}x   "
          f"{iterations_for_span(float(span)):9d}   {in_set(frame).mean():7.1%}")

In [ ]:
# The animated zoom. Each frame is rendered, then stored as a compressed JPEG rather than
# as a grid of numbers: about 40 KB instead of roughly 1 MB, which is the difference
# between a page that loads and one that does not.
import base64, io
from PIL import Image

region_spec = REGIONS["deep_spiral"]
frame_spans = zoom_spans(REGIONS["overview"]["span"], DEEPEST_USEFUL_SPAN, 28)

sources = []
for span in frame_spans:
    frame = render(center=region_spec["center"], span=float(span),
                   width=560, height=420,
                   max_iterations=iterations_for_span(float(span)))
    buffer = io.BytesIO()
    Image.fromarray(to_rgb(frame)).save(buffer, format="JPEG", quality=82)
    sources.append("data:image/jpeg;base64," +
                   base64.b64encode(buffer.getvalue()).decode())

print(f"{len(sources)} frames, {sum(len(s) for s in sources) / 1e6:.1f} MB encoded")

In [ ]:
labels = [f"{REGIONS['overview']['span'] / span:,.0f}x" for span in frame_spans]
zoom_frames = [go.Frame(data=[go.Image(source=sources[k], hoverinfo="skip")],
                        name=labels[k]) for k in range(len(sources))]

zoom = go.Figure(data=[go.Image(source=sources[0], hoverinfo="skip")], frames=zoom_frames)
zoom.update_layout(
    title=f"Zooming into {region_spec['description']}",
    height=600, margin=dict(l=0, r=0, t=48, b=0),
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.06, xanchor="left",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=240, redraw=True), mode="immediate")]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ])],
    sliders=[dict(active=0, x=0.14, len=0.82, y=0.06,
        currentvalue=dict(prefix="Magnification: "),
        steps=[dict(label=f.name, method="animate",
                    args=[[f.name], dict(frame=dict(duration=0, redraw=True),
                                         mode="immediate")]) for f in zoom_frames])],
)
zoom

The final frame magnifies by more than a hundred billion. Scrub back and forth.

- Find a shape that recurs at more than one magnification. **TODO**
- The set is *not* exactly self-similar — the repeats are distorted, not copies. Find a
  repeat that is visibly different from its parent. **TODO**
- The interior fraction in the table above drops to zero after a few frames, yet the
  frames stay interesting. What does that tell you about which measure — interior
  fraction or contrast — actually says whether a view is worth looking at? **TODO**

## 5. Where the arithmetic runs out

Keep zooming and the picture eventually goes flat. That is not the set running out of
detail — the set has detail at every scale. It is the arithmetic.

A `float64` stores about 16 significant decimal digits. Once one pixel is narrower than the
gap between representable numbers near the view's centre, neighbouring pixels round to the
*same* complex number. `smallest_reliable_span` predicts where that happens.

In [ ]:
width = 560
center = REGIONS["deep_spiral"]["center"]
limit = smallest_reliable_span(width, center)

print(f"gap between representable numbers near the centre: {np.spacing(1.0):.2e}")
print(f"predicted breakdown span for a {width}-pixel view: {limit:.2e}")
print()
print("   span      contrast   distinct values")
for span in (1e-11, 1e-12, limit, 1e-14, 1e-15):
    frame = render(center=center, span=float(span), width=width, height=420,
                   max_iterations=iterations_for_span(float(span)))
    finite_frame = frame[np.isfinite(frame)]
    contrast = float(np.std(finite_frame)) if finite_frame.size else 0.0
    distinct = len(np.unique(finite_frame)) / max(finite_frame.size, 1)
    print(f"{span:9.2e}   {contrast:8.2f}   {distinct:14.0%}")

Note which column tells the truth. The fraction of *distinct* values stays at 100% well
past the point where the image is visibly flat, because two pixels differing in the last
bit still count as distinct. **Contrast** is the honest measure.

- At which span does contrast collapse, and how does that compare with the prediction? **TODO**
- Every deep-zoom video you have seen goes far past `1e-13`. What must they be doing
  differently? **TODO**
- This failure is silent: no error, no warning, just a wrong picture. Name one check you
  could add that would catch it. **TODO**

Silent numerical failure is the transferable lesson. The oxygen field in notebook 045 had a
misspelled `_Fillvalue` that xarray ignored without complaint; this has a zoom limit that
no exception announces. Both produce output that looks entirely reasonable.

## 6. Make one of your own

Pick a spot on the boundary in the overview figure, read off approximate coordinates, and
zoom in. You will need to adjust the centre by trial and error — that is the normal
experience, and it is why `REGIONS` exists.

Requirements: state your centre and final span, keep the iteration budget honest, and stop
before `smallest_reliable_span`.

In [ ]:
my_center = (-0.75, 0.1)    # TODO: change me
my_span = 0.05              # TODO: change me

assert my_span > smallest_reliable_span(560, my_center), "past the precision limit"

mine = render(center=my_center, span=my_span, width=560, height=420,
              max_iterations=iterations_for_span(my_span))

escaped_here = mine[np.isfinite(mine)]
interior_fraction = in_set(mine).mean()
contrast = float(np.std(escaped_here)) if escaped_here.size else 0.0

print(f"interior {interior_fraction:.1%}, contrast {contrast:.1f}")
if contrast < 1.0:
    print("Almost no contrast: you are inside the set or far outside it, "
          "not on the boundary. Move the centre.")
else:
    print("Good contrast, so the view straddles the boundary.")

figure = go.Figure(go.Image(z=to_rgb(mine)))
figure.update_layout(title=f"centre {my_center}, span {my_span:g}",
                     height=520, margin=dict(l=0, r=0, t=48, b=0),
                     xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"))
figure

## 7. What the picture is and is not

1. **It is an approximation with a stated budget.** Every black pixel means "did not escape
   within N iterations", not "is in the set". **TODO**
2. **The boundary is where all the cost is.** Interior and far exterior are cheap; the
   filaments are where the iterations go. **TODO**
3. **The colours are contours, not measurements.** Nobody reads a value off this image. **TODO**
4. **The zoom limit is a property of `float64`, not of the set.** **TODO**

**Exit ticket.** In two sentences: this notebook and notebook 045 both contain a failure
that produces a plausible-looking wrong answer with no error message. Name both, and say
what they have in common.